## AI4Climate ML tutorial - Inference and Visualisation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered
In the previous notebooks we have explored and prepared a dataset for training a machine learning model, then we have trained a few different algorithms using this data. The next step is to use the trained model,  both to evaluate and understand how well it has learned the relationship in the data we want it to learn, but also then aplying to the intended use of the data. For example if we have trained a global climate model, we want to use the trained model for experiments around climate change and climate variability, for example.  In this notebook we will look at running inference with the model and visualising the results.


### Prerequisites 
- Same as previous notebooks
- Have completed model training pipeline notebook


### Learning outcomes from completing the notebook
* Load a saved model
* Make predictions with the model
* Visualise the results

## Tutorial 
a balance of explanation and activity



In [2]:
import pathlib
import os
import datetime
import json

In [3]:
import pickle

In [4]:
import numpy

In [ ]:
import pandas

In [ ]:
import iris
import cartopy.crs

In [ ]:
import matplotlib.pyplot

In [ ]:
import sklearn
import sklearn.preprocessing
import sklearn.tree

In [ ]:
import xarray

In [ ]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

In [ ]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [ ]:
current_platform = tutorial_config['platform']

In [ ]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

In [ ]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

In [ ]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [ ]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for inference

In [ ]:
current_res = 1.0

In [ ]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

In [ ]:
zones_df = pandas.read_csv(mlready_data_path)

In [ ]:
# reducing the total data point to decrease memeory requirements
# zones_df = zones_df[(zones_df['period_start']==1991)&(zones_df['scenario']=='historic')]

In [ ]:
zones_df

In [ ]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [ ]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

In [ ]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [ ]:
random_seed = tutorial_config['random_seed']

In [ ]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [ ]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [ ]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


In [ ]:
with open('stats.json','r') as stats_file:
    stats_dict = json.load(stats_file)


In [ ]:
input_scaler = sklearn.preprocessing.StandardScaler()
input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
input_scaler.scale_ = numpy.array(stats_dict['input_scale'])


In [ ]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [ ]:
train_df[[target_var]].value_counts()

In [ ]:
numpy.array(stats_dict['target_classes']).shape

In [ ]:
target_encoder = sklearn.preprocessing.LabelEncoder()
target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

In [ ]:
y_train = target_encoder.transform(train_df[target_var])
y_val = target_encoder.transform(val_df[target_var])
y_test = target_encoder.transform(test_df[target_var])


In [ ]:
algorithm_name = 'decision_tree'

In [ ]:
%%time
load_pretrain=True
if load_pretrain:
    with open(f'{algorithm_name}.pkl', 'rb') as f:
        dt_clf = pickle.load(f)
else:
    dt_opts = {'max_depth':10, 'min_samples_leaf': 2, 'min_samples_split': 5}
    dt_clf = sklearn.tree.DecisionTreeClassifier(**dt_opts)
    dt_clf.fit(X_train, y_train) 

In [ ]:
dt_clf

In [ ]:
y_pred_train = dt_clf.predict(X_train)
y_pred_val = dt_clf.predict(X_val)

In [ ]:
train_df['climate_group_prediction_dt'] = target_encoder.inverse_transform(y_pred_train)
train_df

In [ ]:
val_df['climate_group_prediction_dt'] = target_encoder.inverse_transform(y_pred_val)

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(12,5))

ax1 = fig1.add_subplot(1,2,1,title='distribution of climate zones')
_ = pandas.DataFrame(val_df['climate_group'].value_counts(sort=False)).sort_values(axis=0,by='climate_group').plot.bar(ax=ax1)

ax1 = fig1.add_subplot(1,2,2,title='distribution of climate zone predictions')
_ = pandas.DataFrame(val_df['climate_group_prediction_dt'].value_counts(sort=False)).sort_values(axis=0,by='climate_group_prediction_dt').plot.bar(ax=ax1)


### Visualise on a map

In [ ]:
climate_group_ds = xarray.open_dataset(root_data_dir / '2071_2099' / 'ssp434' /'koppen_geiger_1p0.nc')['kg_class']
climate_group_ds_pred = xarray.open_dataset(root_data_dir / '2071_2099' / 'ssp434' /'koppen_geiger_1p0.nc')['kg_class']

climate_group_ds

In [ ]:
scenario_df = zones_df[(zones_df['scenario'] == 'ssp434') & (zones_df['period_start'] == 2071)]

In [ ]:
scenario_df['climate_group_pred_dt'] = target_encoder.inverse_transform(dt_clf.predict(input_scaler.transform(scenario_df[predictors])))
scenario_df['climate_group_pred_dt_raw'] = dt_clf.predict(input_scaler.transform(scenario_df[predictors]))

In [ ]:
climate_group_ds_pred.data =numpy.zeros(climate_group_ds.data.shape)

In [ ]:
for row1 in scenario_df.iterrows():
    # print(row1[1]['climate_group_pred_dt'])
    climate_group_ds_pred.loc[{'lat':float(row1[1]['lat']), 'lon':float(row1[1]['lon'])} ] = int(row1[1]['climate_group_pred_dt_raw'])
    

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(16,20))
ax1 = fig1.add_subplot(2,1,1 ,projection=cartopy.crs.PlateCarree(),)
climate_group_ds.plot.contourf(ax=ax1, 
                               transform=cartopy.crs.PlateCarree(),
                               cbar_kwargs={"location": "bottom"},
                              )
ax1.coastlines()
ax1.set_title(f'KG CLimate Zones truth')

ax1 = fig1.add_subplot(2,1,2 ,projection=cartopy.crs.PlateCarree(),)
climate_group_ds_pred.plot.contourf(ax=ax1, 
                                    transform=cartopy.crs.PlateCarree(),
                                    cbar_kwargs={"location": "bottom"},
                                   )
ax1.coastlines()
ax1.set_title(f'KG CLimate Zones predicted')

### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)